# W4 작은 예시 노트북 — 단계별 mini 예제

본 노트북(`W4_architectures.ipynb`)이 어렵게 느껴질 때, 한 개념씩 떼어 확인하는 작은 예제 모음이다.
위에서부터 순서대로 실행한다. (`dr_utils.py`, `model_utils.py` 가 같은 폴더에 있어야 한다)

## 예제 1 — softmax 손계산 (numpy)

softmax 는 점수 목록을 "합=1 가중치"로 바꾼다: 각 점수에 exp 를 취하고 전체 합으로 나눈다.
아래 계산을 손으로도 따라 해 본다. e²·⁰≈7.39, e⁰·⁵≈1.65, e⁻¹·⁰≈0.37, 합≈9.41.

In [ ]:
import numpy as np

s = np.array([2.0, 0.5, -1.0])
e = np.exp(s)
w = e / e.sum()
print('exp   =', e.round(2))
print('합    =', e.sum().round(2))
print('softmax =', w.round(2), ' 합 =', w.sum())

## 예제 2 — attention 을 한 단계씩

본 노트북 §2와 같은 예제(토큰 3개, d=2)를 numpy 로 한 단계씩 실행한다.
각 print 결과를 §2의 결과와 비교해 본다.

In [ ]:
X = np.array([[1., 0.], [0., 1.], [1., 1.]])   # 토큰 3개
V = np.array([[10., 0.], [0., 10.], [5., 5.]])

scores = X @ X.T                       # 1) 유사도 (Q=K=X)
scaled = scores / np.sqrt(2)           # 2) ÷ √d
e = np.exp(scaled)
weights = e / e.sum(axis=1, keepdims=True)   # 3) 행별 softmax
out = weights @ V                      # 4) 가중 평균

print('scores =\n', scores)
print('weights =\n', weights.round(2))
print('out =\n', out.round(1))

## 예제 3 — 창 분할의 shape 변화

`window_partition` 은 (B, H, W, C) 토큰 배열을 창 단위로 자른다.
8×8 창이면 64×64 격자가 64개의 창이 된다. 총 토큰 수는 변하지 않는다.

In [ ]:
import torch
from model_utils import window_partition, window_reverse

x = torch.arange(64 * 64, dtype=torch.float32).reshape(1, 64, 64, 1)
win = window_partition(x, 8)
print('분할 전 :', tuple(x.shape), '  토큰', x.numel())
print('분할 후 :', tuple(win.shape), ' 토큰', win.numel(), ' (창 64개 × 8×8)')

back = window_reverse(win, 8, 64, 64)
print('복원 일치 :', bool(torch.equal(x, back)))

## 예제 4 — 2D vs 3D 합성곱 파라미터 세기

커널 크기만 다르고 채널이 같은 두 합성곱의 파라미터를 직접 센다.
3×3=9 대 3×3×3=27, 정확히 3배다.

In [ ]:
import torch.nn as nn

c2 = nn.Conv2d(16, 32, 3, padding=1, bias=False)
c3 = nn.Conv3d(16, 32, 3, padding=1, bias=False)
n2 = sum(p.numel() for p in c2.parameters())
n3 = sum(p.numel() for p in c3.parameters())
print(f'Conv2d 3×3   : {n2:,}  (= 9·16·32)')
print(f'Conv3d 3×3×3 : {n3:,}  (= 27·16·32)')
print(f'비율 = {n3 / n2:.0f}배')